# script for parsing parliament speeches and selecting relevant ones

In [3]:
#imports
import os
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
import csv
from nltk.tokenize import RegexpTokenizer
import re
# from IPython.display import display, HTML
import time   # for automatic pause
# import spacy

tqdm.pandas()
import random
# from IPython.core.display import display, HTML  # for Jupyter Notebook



In [2]:
# #load the large Dutch model
# nlp = spacy.load("nl_core_news_lg")

# classifier 
## code as used before

In [4]:
keywords = (
    "kunstmatige intelligentie|artificial intelligence|artificiële intelligentie|AI|generatieve AI|"
    "generatieve kunstmatige intelligentie|generatieve artificiële intelligentie|"
    "machine learning|machinaal leren|diep leren|deep learning|neurale netwerken|"
    "large language model|grote taalmodel*|LLM|chatbot*|GPT|ChatGPT|Bard|Claude|"
    "Gemini|mistral|perplexity|ollama|LLaMA|openai|anthropic|midjourney|hugging face|"
    "slimme algoritme*|automatische besluitvorming|automatisch beslissysteem|"
    "algoritmische besluitvorming|algoritme*|cognitieve technologie*|AI-technologie*|"
    "AI-systeem*|AI-toepassing*|AI-model*|spraakherkenning|beeldherkenning|"
    "computer vision|natuurlijke taalverwerking|natural language processing|NLP|robot|drones|drone|grok|xai|deepmind|azure"
)

_COMPANY_NAMES = [
    "NVIDIA", "Apple", "Microsoft", "Google", "Alphabet",
    "Meta Platforms", "Facebook", "Tesla", "Oracle",
    "Palantir", "IBM", "Adobe", "Cambricon Technologies",
    "CoreWeave", "Fermi Inc", "Dynatrace", "Tempus AI",
    "SenseTime", "Mobileye", "Aurora Innovation", "UiPath",
    "SoundHound AI", "ASML", "NXP Semiconductors",
    "BE Semiconductor Industries", "ASM International",
    "Adyen", "Just Eat Takeaway", "Booking.com", "Mollie",
    "Picnic", "TomTom", "Swapfiets", "TKH Group",
    "Ordina", "Nedap", "CM.com", "ICT Group",
    "Neways Electronics", "Ctac", "Photon Energy",
    "Almunda Professionals", "Samsung", "Huawei",
    "Sony", "LG", "Baidu", "Tencent",
    "Alibaba", "Douyin", "Cloudflare",
    "Snowflake", "Docker", "Red Hat",
    "Uber", "Bolt", "Grab", "Epic Games",
    "Unity", "Discord", "Twitter", "X"
]

def normalize_keyword(k: str) -> str:
    """Convert wildcard-like keywords into safe, precise regex patterns."""

    k = k.strip().lower()

    # algoritme → algoritme, algoritmen, algoritmes
    if k in {"algoritme", "algoritme*"}:
        return r"algoritm(?:e|en|es)"

    # chatbots
    if k in {"chatbot", "chatbot*"}:
        return r"chatbots?"

    # llm / llms
    if k in {"llm", "llm*"}:
        return r"llms?"

    # grote taalmodel / grote taalmodels
    if k in {"grote taalmodel*"}:
        return r"grote taalmodel(?:s)?"

    # intelligent(e) algoritme(n)
    if "intelligente algoritme" in k:
        return r"intelligent[e]?\s+algoritm(?:e|en|es)?"

    # slimme algoritme(n)
    if "slimme algoritme" in k:
        return r"slimm[e]?\s+algoritm(?:e|en|es)?"

    # ANY OTHER keyword with a trailing "*" should become:
    #   <base> → <base>(?:s)?  
    # But only if safe.
    if k.endswith("*"):
        base = k[:-1]
        # optional plural “s”
        return re.escape(base) + r"s?"

    return re.escape(k)

# Convert into list
keywords = keywords.strip().split('|')
set_ai_words = {k for k in keywords if k.strip()}

_AI_PAT = re.compile(
    r"\b(" + "|".join(normalize_keyword(k) for k in set_ai_words) + r")\b",
    re.IGNORECASE
)

# Weiwei filter
_WEIWEI_PAT = re.compile(r'\bweiwei\b', re.IGNORECASE)

# Pattern that detects the combined form "kunstmatige intelligentie (AI)" ---
_KI_AI_PAT = re.compile(r'kunstmatige\s+intelligentie\s*\(\s*ai\s*\)', re.IGNORECASE)

def _collapse_ki_ai(matches: list[str], text: str) -> list[str]:
    """
    If the text contains 'kunstmatige intelligentie (AI)', remove up to that many 'AI'
    occurrences from the match list so the pair counts as ONE hit.
    """
    n_pairs = len(_KI_AI_PAT.findall(text))
    if n_pairs == 0:
        return matches
    kept, removed = [], 0
    for m in matches:
        if m.lower() == "ai" and removed < n_pairs:
            removed += 1         # drop this 'AI' because it's part of the pair
        else:
            kept.append(m)
    return kept

# --- Remove weiwei articles before classifying ---
def _drop_weiwei_rows(df, title_col='title', body_col='text'):
    mask = (
        df[title_col].astype(str).str.contains(_WEIWEI_PAT, na=False) |
        df[body_col].astype(str).str.contains(_WEIWEI_PAT, na=False)
    )
    removed = mask.sum()
    # print(f"Removed {removed} articles containing 'weiwei'.")
    return df.loc[~mask].copy()

_COMPANY_PAT = re.compile(
    r'\b(' + '|'.join(re.escape(n) for n in _COMPANY_NAMES) + r')\b',
    re.IGNORECASE
)
_COMPANY_TOKENS = {n.lower() for n in _COMPANY_NAMES}  # to compare against matched keyword tokens

# --- 3) Classification ---
def ai_classification(df, title_col='title', body_col='text'):

    # hard remove weiwei articles
    df = _drop_weiwei_rows(df, title_col, body_col)

    labels, matched_title, matched_body, matched_all = [], [], [], []
    n_hits_title_total, n_hits_body_total = [], []
    matched_companies_all = [] #store company hits (per row)

    for _, row in tqdm(df.iterrows(), total=df.shape[0]):
        title = row[title_col] if pd.notna(row[title_col]) else ""
        body  = row[body_col]  if pd.notna(row[body_col])  else ""
        
        # --- collect company hits from raw text (title + body)
        comp_title = _COMPANY_PAT.findall(str(title))
        comp_body  = _COMPANY_PAT.findall(str(body))
        company_hits = sorted(set(m.lower() for m in (comp_title + comp_body)))
        matched_companies_all.append(company_hits)

        # regex matches
        title_matches = _AI_PAT.findall(str(title))
        body_matches  = _AI_PAT.findall(str(body))

        # --- collapse KI (AI) double-counts ---
        title_matches = _collapse_ki_ai(title_matches, title)
        body_matches  = _collapse_ki_ai(body_matches, body)

        # ---  ignore 'claude' if it's the ONLY match across title+body ---
        all_lower = [m.lower() for m in (title_matches + body_matches)]
        if set(all_lower) == {"claude"}:
            title_matches, body_matches = [], []
            all_lower = []

        # decision rule
        title_match = len(title_matches) >= 1
        body_match  = len(body_matches)  >= 2
        label = "yes" if (title_match or body_match) else "no"

        # --- company mention + at least one other keyword -> ai_related = yes ---
        company_present = bool(company_hits)
        other_hits = [m for m in all_lower if m not in _COMPANY_TOKENS]
        if company_present and len(other_hits) >= 1:
            label = "yes"

        labels.append(label)
        matched_title.append(sorted(set(m.lower() for m in title_matches)))
        matched_body.append(sorted(set(m.lower() for m in body_matches)))
        matched_all.append(sorted(set(m.lower() for m in (title_matches + body_matches))))
        n_hits_title_total.append(len(title_matches))
        n_hits_body_total.append(len(body_matches))

    # write back
    df['ai_related'] = labels
    df['matched_keywords_title'] = matched_title
    df['matched_keywords_body']  = matched_body
    df['matched_keywords_all']   = matched_all
    df['n_hits_title_total'] = n_hits_title_total
    df['n_hits_body_total']  = n_hits_body_total
    df['company_hits'] = matched_companies_all

    return df


# selecting relevant articles

In [7]:
import ast 

# --- Functions ---
def split_into_sentences(text):
    if not isinstance(text, str):
        return []
    return re.split(r'(?<=[.!?])[\s\n]+', text)

def count_keywords(sentence, keywords):
    if not keywords:
        return 0
    return sum(1 for keyword in keywords if re.search(fr'\b{re.escape(keyword)}\b', sentence, re.I))

def merge_spans(spans):
    """Merge overlapping or adjacent spans into clusters."""
    if not spans:
        return []
    spans.sort()
    merged = [spans[0]]
    for s, e in spans[1:]:
        last_s, last_e = merged[-1]
        if s <= last_e:  # overlap or adjacency
            merged[-1] = (last_s, max(last_e, e))
        else:
            merged.append((s, e))
    return merged

def extract_relevant_sections(row, text_col='text', keywords_col='matched_keywords_all', context_window=2):
    """
    Extract multiple keyword clusters with +/- context_window sentences.
    Returns blocks separated by blank lines.
    """
    text = row[text_col]
    keywords = row[keywords_col]

    if not isinstance(text, str):
        return ""
    if keywords is None or isinstance(keywords, float):
        keywords = []
    if isinstance(keywords, str):
        keywords = [keywords]

    sentences = split_into_sentences(text)
    if not sentences or not keywords:
        return ""

    relevant_indices = [i for i, s in enumerate(sentences) if count_keywords(s, keywords) > 0]
    if not relevant_indices:
        return ""

    # Build spans around each relevant index
    spans = []
    for i in relevant_indices:
        start = max(0, i - context_window)
        end = min(len(sentences), i + context_window + 1)
        spans.append((start, end))

    # Merge overlapping spans into clusters
    merged_spans = merge_spans(spans)

    # Collect blocks
    blocks = [" ".join(sentences[s:e]) for s, e in merged_spans]
    return "\n\n".join(blocks)

def prepare_for_rel(df):
    df['matched_keywords_all'] = df['matched_keywords_all'].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

    df['company_hits'] = df['company_hits'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

    df['matched_keywords_all'] = df.apply(
    lambda row: (row['company_hits'] or []) + (row['matched_keywords_all'] or []),
    axis=1
    )
    return df

In [8]:
folder_path = r"C:\Users\joly-\Github\HUMAN\tweede_kamer\scraping_plen_ver\plenaire_verslagen_html"

rows = []
html_files = [f for f in os.listdir(folder_path) if f.endswith(".html")]

print(f"Found {len(html_files)} HTML files. Starting classification...")

for idx, filename in enumerate(tqdm(html_files, desc="Parsing HTML files"), start=1):
    file_path = os.path.join(folder_path, filename)

    file_size = os.path.getsize(file_path)
    # if file_size > 50 * 1024 * 1024:  # skip files > 50 MB
    #     print(f"Skipping large file: {filename} ({file_size / 1e6:.1f} MB)")
    #     continue

    with open(file_path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f, 'html.parser')
    
    # Extract paragraphs
    paragraphs = [p.get_text() for p in soup.find_all('p')]
    paragraphs = [p.strip() for p in paragraphs if p.strip()]
    text_with_paragraphs = '\n'.join(paragraphs)
    
    # Create a temporary DataFrame with one row
    temp_df = pd.DataFrame([{'filename': filename, 'text': text_with_paragraphs}])
    
    # Apply classification
    classified_df = ai_classification(temp_df, title_col='filename', body_col='text')
    
    # Keep only if ai_related == 'yes'
    if classified_df['ai_related'].iloc[0] == 'yes':
        print(f"[INFO] AI-related document found: {filename} ({idx}/{len(html_files)})")
        classified_df = prepare_for_rel(classified_df)
        classified_df['relevant_text'] = classified_df.apply(lambda row: extract_relevant_sections(row, text_col='text', keywords_col='matched_keywords_all'), axis=1)
        classified_df = classified_df.drop(columns=['text'])  # drop the full text column to save memory
        rows.append(classified_df.iloc[0])  # append the row
    
# Combine all rows into final DataFrame
df = pd.DataFrame(rows)

print("Done! DataFrame contains", len(df), "documents.")

#save to csv
df.to_csv("plenaire_verslagen_classified.csv", index=False)

Found 1133 HTML files. Starting classification...


100%|██████████| 1/1 [01:16<00:00, 76.04s/it]01:26<27:06:14, 86.20s/it]


[INFO] AI-related document found: kamerstukken-plenaire_verslagen-detail-2014-2015-101_9ae18352.html (2/1133)


100%|██████████| 1/1 [01:50<00:00, 110.10s/it]3:54<23:31:30, 74.95s/it]


[INFO] AI-related document found: kamerstukken-plenaire_verslagen-detail-2014-2015-103_09e01e2b.html (4/1133)


...

[INFO] AI-related document found: kamerstukken-plenaire_verslagen-detail-2024-2025-99_10d11e0d.html (1132/1133)


100%|██████████| 1/1 [00:29<00:00, 29.56s/it]3 [14:38:51<00:38, 38.69s/it]


[INFO] AI-related document found: kamerstukken-plenaire_verslagen-detail-2024-2025-9_e4e03e20.html (1133/1133)


Parsing HTML files: 100%|██████████| 1133/1133 [14:39:38<00:00, 46.58s/it]


Done! DataFrame contains 435 documents.


In [9]:
df.info()

<class 'pandas.DataFrame'>
Index: 435 entries, 0 to 0
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   filename                435 non-null    str   
 1   ai_related              435 non-null    str   
 2   matched_keywords_title  435 non-null    object
 3   matched_keywords_body   435 non-null    object
 4   matched_keywords_all    435 non-null    object
 5   n_hits_title_total      435 non-null    int64 
 6   n_hits_body_total       435 non-null    int64 
 7   company_hits            435 non-null    object
 8   relevant_text           435 non-null    str   
dtypes: int64(2), object(4), str(3)
memory usage: 34.0+ KB
